In [61]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:20pt;}
.inner_cell{font-size:20pt;}
div.text_cell_render pre code {font-size:20pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:20pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:20pt;padding:5px;}
table.dataframe{font-size:18px;}
</style>
"""))

<font color="red" size="6"><b>ch14. 웹 데이터 수집</b></font>

# 1절. BeautifulSoup과 parser
    (정적 웹크롤링, 공공api사용)

`pip install bs4` 아나콘다를 설치하면 자동 설치되는 패키지에 포함
- 공식 사이트 : https://www.crummy.com/software/BeautifulSoup/
- documentation :  https://www.crummy.com/software/BeautifulSoup/bs4/doc/

In [7]:
import requests # HTTP 요청 처리하는 lib
# file:// == c:
# http://www.
from requests_file import FileAdapter

In [11]:
# 로컬에 있는 파일을 웹요청하듯이 읽어오는 작업
s = requests.Session();
s.mount("file://", FileAdapter()) # file://로 시작하는 url을 어댑터가 처리
response = s.get("file:///ai/lecNote/01_python/data/ch14_sample.html") # c:/ai/lecNote/data/ch14_sample.html
response

<Response [200]>

In [12]:
if response:
    print('해당 url에 접근함')
else:
    print('해당 url에 거부됨')

해당 url에 접근함


In [13]:
response.status_code
# 200 : 정상
# 404 : 없는 페이지

200

In [15]:
response.content # html의 바이너리 형식의 내용

b'<!DOCTYPE html>\r\n<html lang="en">\r\n<head>\r\n  <meta charset="UTF-8">\r\n</head>\r\n<body>\r\n  <h1 class="greeting css" id="text">Hello, CSS</h1>\r\n  <h1 class="css">Hi, CSS</h1>\r\n  <div id="subject">subject \xec\x84\xa0\xed\x83\x9d\xec\x9e\x90 \xec\x95\x88\xec\x9d\x98 \xeb\x82\xb4\xec\x9a\xa9</div>\r\n  <p>CSS \xec\x84\xa0\xed\x83\x9d\xec\x9e\x90\xeb\x8a\x94 \xeb\x8b\xa4\xec\x96\x91\xed\x95\x9c \xea\xb3\xb3\xec\x97\x90\xec\x84\x9c \xed\x99\x9c\xec\x9a\xa9\xeb\x90\xa9\xeb\x8b\x88\xeb\x8b\xa4</p>\r\n  <div class="contents">\r\n    \xec\x84\xa0\xed\x83\x9d\xec\x9e\x90\xeb\xa5\xbc \xec\x96\xb4\xeb\x96\xbb\xea\xb2\x8c \xec\x9e\x91\xec\x84\xb1\xed\x95\x98\xeb\x8a\x90\xeb\x83\x90\xec\x97\x90 \xeb\x94\xb0\xeb\x9d\xbc\r\n    <span>\xeb\x8b\xa4\xeb\xa5\xb8<b>\xec\x9a\x94\xec\x86\x8c\xea\xb0\x80 \xeb\xb0\x98\xed\x99\x98</b></span>\xeb\x90\xa9\xeb\x8b\x88\xeb\x8b\xa4\r\n  </div>\r\n  <div>CSS \xec\x84\xa0\xed\x83\x9d\xec\x9e\x90\xeb\x8a\x94 \xeb\x8b\xa4\xec\x96\x91\xed\x95\x9c \xea\xb3\

In [16]:
print(response.content.decode('utf-8'))

<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
</head>
<body>
  <h1 class="greeting css" id="text">Hello, CSS</h1>
  <h1 class="css">Hi, CSS</h1>
  <div id="subject">subject 선택자 안의 내용</div>
  <p>CSS 선택자는 다양한 곳에서 활용됩니다</p>
  <div class="contents">
    선택자를 어떻게 작성하느냐에 따라
    <span>다른<b>요소가 반환</b></span>됩니다
  </div>
  <div>CSS 선택자는 다양한 곳에 <b>활용</b>됩니다</div>
</body>
</html>


In [19]:
response.text

'<!DOCTYPE html>\r\n<html lang="en">\r\n<head>\r\n  <meta charset="UTF-8">\r\n</head>\r\n<body>\r\n  <h1 class="greeting css" id="text">Hello, CSS</h1>\r\n  <h1 class="css">Hi, CSS</h1>\r\n  <div id="subject">subject 선택자 안의 내용</div>\r\n  <p>CSS 선택자는 다양한 곳에서 활용됩니다</p>\r\n  <div class="contents">\r\n    선택자를 어떻게 작성하느냐에 따라\r\n    <span>다른<b>요소가 반환</b></span>됩니다\r\n  </div>\r\n  <div>CSS 선택자는 다양한 곳에 <b>활용</b>됩니다</div>\r\n</body>\r\n</html>'

In [22]:
# html 파싱 객체
from bs4 import BeautifulSoup
soup = BeautifulSoup(response.text, # response.content,
                    "html.parser")
# soup

In [29]:
# 1. soup.select_one('선택자') : 해당 선택자 처음 하나 엘리먼트만
el = soup.select_one('h1.css')
print('el =', el)
print('el.text   =>', el.text)
print('el.string =>', el.string)
print('el의 속성들 =>', el.attrs)
print('el의 class 속성 =>', el.attrs['class'])
print('el의 class 속성 =>', el.attrs.get('class'))
# print('el의 href 속성(없는 속성은 에러) =>', el.attrs.get['href'])
print('el의 href 속성 =>', el.attrs.get('href'))
print('el의 name =>', el.name)

el = <h1 class="greeting css" id="text">Hello, CSS</h1>
el.text   => Hello, CSS
el.string => Hello, CSS
el의 속성들 => {'class': ['greeting', 'css'], 'id': 'text'}
el의 class 속성 => ['greeting', 'css']
el의 class 속성 => ['greeting', 'css']
el의 href 속성 => None
el의 name => h1


In [31]:
# 2. soup.select('선택자') : 해당 선택자 엘리먼트 다 list로
els = soup.select('h1.css')
print('els =>', els)
print('els들의 text =>', [el.text for el in els])
print('els들의 string =>', [el.string for el in els])
print('els들의 속성들 =>', [el.attrs for el in els])
print('els들의 class 속성 =>', [el.attrs.get('class') for el in els])

els => [<h1 class="greeting css" id="text">Hello, CSS</h1>, <h1 class="css">Hi, CSS</h1>]
els들의 text => ['Hello, CSS', 'Hi, CSS']
els들의 string => ['Hello, CSS', 'Hi, CSS']
els들의 속성들 => [{'class': ['greeting', 'css'], 'id': 'text'}, {'class': ['css']}]
els들의 class 속성 => [['greeting', 'css'], ['css']]


In [34]:
# 3. soup.find(태그, 속성) vs soup.select_one('선택자') : 해당 속성을 갖는 태그 처음 하나만
print('select_one :', soup.select_one('h1.css'))
print('find       :', soup.find('h1', {'class':'css'}))
print('find       :', soup.find('h1', class_='css'))
print()
print('select_one :', soup.select_one('h1#text'))
print('select_one :', soup.find('h1', {'id':'text'}))

select_one : <h1 class="greeting css" id="text">Hello, CSS</h1>
find       : <h1 class="greeting css" id="text">Hello, CSS</h1>
find       : <h1 class="greeting css" id="text">Hello, CSS</h1>

select_one : <h1 class="greeting css" id="text">Hello, CSS</h1>
select_one : <h1 class="greeting css" id="text">Hello, CSS</h1>


In [37]:
# 4. soup.find_all(태그, 속성) vs. soup.select('선택자') : 해당 엘리먼트 다 list로
print('모든 h1.css와 span태그 :', soup.select('h1.css, span'))
print('모든 h1.css와 span태그 :', soup.find_all(['h1'], class_='css') + 
                                soup.find_all('span'))

모든 h1.css와 span태그 : [<h1 class="greeting css" id="text">Hello, CSS</h1>, <h1 class="css">Hi, CSS</h1>, <span>다른<b>요소가 반환</b></span>]
모든 h1.css와 span태그 : [<h1 class="greeting css" id="text">Hello, CSS</h1>, <h1 class="css">Hi, CSS</h1>, <span>다른<b>요소가 반환</b></span>]


In [41]:
# 없는 엘리먼트 찾기
print('find_all(빈 list) :', soup.find_all('a'))
print('find(None)        :', soup.find('a'))
print('select(빈 list)   :', soup.select('a'))
print('select_one(None)  :', soup.select_one('a'))

find_all(빈 list) : []
find(None)        : None
select(빈 list)   : []
select_one(None)  : None


# 2절. 정적 웹 데이터 수집(정적 웹크롤링)
## 2.1 BeautifulSoup 모듈을 활용한 html 웹 데이터 수집
### 1) 환율정보 가져오기(네이버증권>시장지표)

- https://finance.naver.com/marketindex/

    * 크롤링 허용범위는 사이트마다 ~/robots.txt에서 확인할 수 있음
        - Allow : 크롤링 허용가능한 폴더
        - Disallow : 크롤링 제한 폴더

In [3]:
# soup 객체 생성 방법 1
import requests 
from bs4 import BeautifulSoup
url = 'https://finance.naver.com/marketindex/'
response = requests.get(url)
# response # Response
print(response.status_code)
# response.text # response.content
soup = BeautifulSoup(response.text, 'html.parser')

200


In [4]:
# soup 객체 생성 방법 2
from urllib.request import urlopen
url = 'https://finance.naver.com/marketindex/'
response = urlopen(url)
# response # HTTPResponse
print(response.status)
# print(response.read().decode('cp949'))
soup = BeautifulSoup(response, 'html.parser')
soup

200



<script language="javascript" src="/template/head_js.naver?referer=info.finance.naver.com&amp;menu=marketindex&amp;submenu=market"></script>
<script src="https://ssl.pstatic.net/imgstock/static.pc/20260731153938/js/info/jindo.min.ns.1.5.3.euckr.js" type="text/javascript"></script>
<script src="https://ssl.pstatic.net/imgstock/static.pc/20260731153938/js/jindo.1.5.3.element-text-patch.js" type="text/javascript"></script>
<div id="container" style="padding-bottom:0px;">
<div class="market_include">
<div class="market_data">
<div class="market1">
<div class="title">
<h2 class="h_market1"><span>환전 고시 환율</span></h2>
</div>
<!-- data -->
<div class="data">
<ul class="data_lst" id="exchangeList">
<li class="on">
<a class="head usd" href="/marketindex/exchangeDetail.naver?marketindexCd=FX_USDKRW" onclick="clickcr(this, 'fr1.usdt', '', '', event);">
<h3 class="h_lst"><span class="blind">미국 USD</span></h3>
<div class="head_info point_up">
<span class="value">1,416.20</span>
<span class="txt_krw

In [5]:
p = '1,417,000.70'
float(p.replace(',','')) # 방법1
float(''.join(p.split(','))) # 방법2

1417000.7

In [6]:
# div.head_info 밑의 span.value (find계열)
prices = []
headinfos = soup.find_all('div', class_='head_info')
for headinfo in headinfos:
    # print(headinfo)
    price = headinfo.find('span', class_='value')
    prices.append(float(''.join(price.text.split(','))))
print(prices)

[1416.2, 888.43, 1633.3, 209.87, 159.24, 1.1541, 1.3507, 99.71, 83.2, 1863.86, 4441.1, 199692.77]


In [7]:
# span.value (find계열)
price_els = soup.find_all('span', class_='value')
prices = [round(float(price.text.replace(',','')), 1) for price in price_els]
print(prices)

[1416.2, 888.4, 1633.3, 209.9, 159.2, 1.2, 1.4, 99.7, 83.2, 1863.9, 4441.1, 199692.8]


In [52]:
# 금액들 : div.head_info 밑의 span.value
price_els = soup.select('div.head_info > span.value')
# price_els

[<span class="value">1,417.70</span>,
 <span class="value">889.17</span>,
 <span class="value">1,635.60</span>,
 <span class="value">210.09</span>,
 <span class="value">159.2400</span>,
 <span class="value">1.1541</span>,
 <span class="value">1.3507</span>,
 <span class="value">99.7100</span>,
 <span class="value">83.2</span>,
 <span class="value">1863.92</span>,
 <span class="value">4441.1</span>,
 <span class="value">200855.05</span>]

In [8]:
# 차이틀
title_els = soup.select('h3.h_lst > span.blind')
# title_els

In [12]:
# 단위들 : div.head_info span.blind
unit_els = soup.select('div.head_info > span > span.blind')
len(unit_els)
units = [unit_el.string for unit_el in unit_els]
units.insert(7, '')
units

['원', '원', '원', '원', '엔', '달러', '달러', '', '달러', '원', '달러', '원']

In [13]:
# 상승/하락 : div.head_info > span.blind
trend_els = soup.select('div.head_info > span.blind')
len(trend_els)

12

In [14]:
len(title_els), len(price_els), len(units), len(trend_els)

(12, 12, 12, 12)

In [15]:
for idx in range(len(title_els)):
    print('{} : {} {} - {}'.format(title_els[idx].text,
                                   price_els[idx].text,
                                   units[idx],
                                   trend_els[idx].text))

미국 USD : 1,416.20 원 - 상승
일본 JPY(100엔) : 888.43 원 - 상승
유럽연합 EUR : 1,633.30 원 - 상승
중국 CNY : 209.87 원 - 상승
달러/일본 엔 : 159.2400 엔 - 상승
유로/달러 : 1.1541 달러 - 하락
영국 파운드/달러 : 1.3507 달러 - 하락
달러인덱스 : 99.7100  - 상승
WTI : 83.2 달러 - 상승
휘발유 : 1863.86 원 - 하락
국제 금 : 4441.1 달러 - 상승
국내 금 : 199692.77 원 - 상승


In [16]:
for title, price, unit, trend in zip(title_els, price_els, units, trend_els):
    print("{} : {}{} - {}".format(title.text, price.text, unit, trend.text))

미국 USD : 1,416.20원 - 상승
일본 JPY(100엔) : 888.43원 - 상승
유럽연합 EUR : 1,633.30원 - 상승
중국 CNY : 209.87원 - 상승
달러/일본 엔 : 159.2400엔 - 상승
유로/달러 : 1.1541달러 - 하락
영국 파운드/달러 : 1.3507달러 - 하락
달러인덱스 : 99.7100 - 상승
WTI : 83.2달러 - 상승
휘발유 : 1863.86원 - 하락
국제 금 : 4441.1달러 - 상승
국내 금 : 199692.77원 - 상승


In [20]:
import pandas as pd;
data = []
for title, price, print, trend in zip(title_els, price_els, units, trend_els):
    data.append({'title' : title.text, 
                 'price' : float(price.text.replace(',','')),
                 'unit' : unit,
                 'trend' : trend.text})
pd.DataFrame(data) #. to_csv('data/file.csv', index=False)

,title,price,unit,trend
0,미국 USD,1416.2000,원,상승
1,일본 JPY(100엔),888.4300,원,상승
2,유럽연합 EUR,1633.3000,원,상승
3,중국 CNY,209.8700,원,상승
4,달러/일본 엔,159.2400,원,상승
5,유로/달러,1.1541,원,하락
6,영국 파운드/달러,1.3507,원,하락
7,달러인덱스,99.7100,원,상승
8,WTI,83.2000,원,상승
9,휘발유,1863.8600,원,하락


In [22]:
data = []
for title, price, print, trend in zip(title_els, price_els, units, trend_els):
    data.append([title.text, float(price.text.replace(',','')), unit, trend.text])
pd.DataFrame(data, columns=['title','price','unit','trend'])

,title,price,unit,trend
0,미국 USD,1416.2000,원,상승
1,일본 JPY(100엔),888.4300,원,상승
2,유럽연합 EUR,1633.3000,원,상승
3,중국 CNY,209.8700,원,상승
4,달러/일본 엔,159.2400,원,상승
5,유로/달러,1.1541,원,하락
6,영국 파운드/달러,1.3507,원,하락
7,달러인덱스,99.7100,원,상승
8,WTI,83.2000,원,상승
9,휘발유,1863.8600,원,하락


### 2) 이번주 로또번호 출력
- 방법2에서 User Agent를 추가하여 soup생성
- https://search.daum.net/search?w=tot&DA=YZR&t__nil_searchbox=btn&q=lotto (다음에서 lotto검색)
```
   1236회(2026.08.08 추첨)
   당첨번호 [12, 18, 21, 29, 34, 38]
   보너스 10
```

In [10]:
# 방법 1 
import requests
from bs4 import BeautifulSoup
url = 'https://search.daum.net/search?w=tot&DA=YZR&t__nil_searchbox=btn&q=lotto'
response = requests.get(url)
print('response의 상태 :',response.status_code)
soup = BeautifulSoup(response.text, 'html.parser')
# soup

response의 상태 : 200


In [9]:
# 방법 2
from urllib.request import urlopen, Request
headers = {'User-Agent':
          'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36'}
request = Request(url, headers=headers)
response = urlopen(request)
print('response의 상태 :', response.status)
soup = BeautifulSoup(response, 'html.parser')
# soup

response의 상태 : 200


In [23]:
#  1236회 (2026.08.08 추첨)
#  당첨번호 [12, 18, 21, 29, 34, 38]
#  보너스 10
times = soup.select_one('div.prize span.f_red').text
date = soup.select_one('div.prize > span.date').text
title1 = soup.select_one('div.prize > strong').text[-4:]
lotto_numbers = soup.select('div.lottonum > span.ball:nth-last-child(n+3)')
lotto_numbers = soup.select('div.lottonum > span.ball')[:-2]
title2 = soup.select_one('div.lottonum span.screen_out').text
bonus_number = soup.select_one('div.lottonum > span.bg_ball1').text
print(times, date)
print(title1, [int(numbers.text) for numbers in lotto_numbers])
print(title2, bonus_number)

1236회 (2026.08.08 추첨)
당첨번호 [12, 18, 21, 29, 34, 38]
보너스 10


In [26]:
# 위의 select계열 함수를 find계열함수로 변경하여 구연해보기
# times = soup.select_one('div.prize span.f_red').text
prize = soup.find('div', class_='prize')
times = prize.find('span', class_='f_red').text

# date = soup.select_one('div.prize > span.date').text
date = prize.find('span', class_='date').text

# title1 = soup.select_one('div.prize > strong').text[-4:]
title1 = prize.find('strong').text[-4:]

# lotto_numbers = soup.select('div.lottonum > span.ball')[:-2]
lottonum = soup.find('div', class_='lottonum')
lotto_numbers = lottonum.find_all('span', class_='ball')[:-2]

# title2 = soup.select_one('div.lottonum span.screen_out').text
title2 = lottonum.find('span', class_='screen_out').text

# bonus_number = soup.select_one('div.lottonum > span.bg_ball1').text
bonus_number = lottonum.find('span', class_='bg_ball1').text

print(times, date)
print(title1, [int(numbers.text) for numbers in lotto_numbers])
print(title2, bonus_number)

1236회 (2026.08.08 추첨)
당첨번호 [12, 18, 21, 29, 34, 38]
보너스 10


### 3) 다음 뉴스 검색 리스트
```
no title    href
0  타이틀 1  http://~
1  타이틀 2  http://~
2  타이틀 3  http://~
```

In [4]:
# 방법 1
import requests
from bs4 import BeautifulSoup
word = '처서'
url = f'https://search.daum.net/search?nil_suggest=btn&w=news&DA=SBC&cluster=y&q={word}'
print(url)
response = requests.get(url)
print(response.status_code)
soup = BeautifulSoup(response.text, 'html.parser')

https://search.daum.net/search?nil_suggest=btn&w=news&DA=SBC&cluster=y&q=처서
200


In [5]:
# 방법 2 
from urllib.request import urlopen, Request
from urllib.parse import quote
word = quote('처서')
url = f'https://search.daum.net/search?nil_suggest=btn&w=news&DA=SBC&cluster=y&q={word}' 
print(url)
headers = {'User-Agent':
          'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36'}
# request = Request(url, headers=headers)
request = Request(url)
request.add_header('User-Agent',
                  'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36')
response = urlopen(request)
print(response.status)
soup = BeautifulSoup(response, 'html.parser')
# soup

https://search.daum.net/search?nil_suggest=btn&w=news&DA=SBC&cluster=y&q=%EC%B2%98%EC%84%9C
200


In [6]:
items_find_list = [] # 검색한 결과를 담을 dict 리스트
items_el = soup.select('div.item-title > strong.tit-g > a')
for idx, item in enumerate (items_el):
    # print(idx, item)
    items_find_list.append({'no':idx,
                            'title':item.text,
                            'link':item.attrs.get('href')})
import pandas as pd
pd.DataFrame(items_find_list)

,no,title,link
0,0,[길섶에서] 입추와 처서 사이,http://v.daum.net/v/20260811051211434
1,1,쪽빛 손수건 물들이며 처서(處暑) 정취 느껴요,http://v.daum.net/v/20260812142747032
2,2,"‘처서 매직’ 옛말…9월까지 열대야, 한반도 여름 길고 독해졌다",http://v.daum.net/v/20260724112507369
3,3,“희경루서 처서에 즐기는 천연염색·남도 음식 체험”,http://v.daum.net/v/20260812185141950
4,4,"40도 폭염은 꺾였는데…처서까지 가뭄 풀릴까, '찬홈'이 변수",http://v.daum.net/v/20260811063840577
5,5,"가족과 함께 24절기 즐겨요…성북, ‘절기야 놀자’ 입추·처서 편 운영",http://v.daum.net/v/20260812112507609
6,6,"[언중언]입추, 처서 그리고 태풍",http://v.daum.net/v/20260813000135309
7,7,"""처서까지 비 안 오면 농사 망칠 것""",http://v.daum.net/v/20260810210754818
8,8,"[르포] ""처서까지 비 안 오면 농사 포기할 판""…바짝 타는 경남 농심",http://v.daum.net/v/20260810145850338
9,9,"“처서가 오면 상처도 마른다”…재난과 상처, 견뎌냄의 지혜 배운다",http://v.daum.net/v/20260803093647511


In [7]:
items_find_list = [] # 검색한 결과를 담은 2차원 리스트
items_el = soup.select('div.item-title > strong.tit-g > a')
for idx, item in enumerate(items_el):
    items_find_list.append([idx, item.text, item.attrs.get('href')])
pd.DataFrame(items_find_list, columns=['순번', '기사제목', '링크'])

,순번,기사제목,링크
0,0,[길섶에서] 입추와 처서 사이,http://v.daum.net/v/20260811051211434
1,1,쪽빛 손수건 물들이며 처서(處暑) 정취 느껴요,http://v.daum.net/v/20260812142747032
2,2,"‘처서 매직’ 옛말…9월까지 열대야, 한반도 여름 길고 독해졌다",http://v.daum.net/v/20260724112507369
3,3,“희경루서 처서에 즐기는 천연염색·남도 음식 체험”,http://v.daum.net/v/20260812185141950
4,4,"40도 폭염은 꺾였는데…처서까지 가뭄 풀릴까, '찬홈'이 변수",http://v.daum.net/v/20260811063840577
5,5,"가족과 함께 24절기 즐겨요…성북, ‘절기야 놀자’ 입추·처서 편 운영",http://v.daum.net/v/20260812112507609
6,6,"[언중언]입추, 처서 그리고 태풍",http://v.daum.net/v/20260813000135309
7,7,"""처서까지 비 안 오면 농사 망칠 것""",http://v.daum.net/v/20260810210754818
8,8,"[르포] ""처서까지 비 안 오면 농사 포기할 판""…바짝 타는 경남 농심",http://v.daum.net/v/20260810145850338
9,9,"“처서가 오면 상처도 마른다”…재난과 상처, 견뎌냄의 지혜 배운다",http://v.daum.net/v/20260803093647511


In [10]:
# 다음 뉴스 검색 함수(원하는 키워드, 원하는 페이지로)
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
def collect_list(keyword, page):
    'keyword로 해당 page에 검색한 결과 dict list return'
    # url = f'https://search.daum.net/search?w=news'
    url = 'https://search.daum.net/search?w=news'
    params = {'q':keyword, 'p':page}
    response = requests.get(url, params=params)
    soup = BeautifulSoup(response.text, 'html.parser')
    items_find_list = []
    items_el = soup.select('div.item-title > strong.tit-g > a')
    for idx, item in enumerate(items_el):
        items_find_list.append({'no':(page-1)*10 + idx, 
                                'title':item.text,
                                'link':item.attrs.get('href')})
    return items_find_list

In [11]:
collect_list('영화', 2)

[{'no': 10,
  'title': ' GICON, 영화음악 크리에이터 양성 최대 5팀 모집 ',
  'link': 'http://v.daum.net/v/20260812150451141'},
 {'no': 11,
  'title': ' 제천국제음악영화제 2026, 영화와 함께 밤새는 ‘녹턴 스페셜’ 첫선 ',
  'link': 'http://v.daum.net/v/20260813111039395'},
 {'no': 12,
  'title': ' 영화 흥행에 단종 유배지 ‘돌탑 물결’…청령포에 소원이 쌓인다 ',
  'link': 'http://v.daum.net/v/20260813153009929'},
 {'no': 13,
  'title': " 7일 만에 예고편 '1000만'→1761만 국민 배우 자존심 건 韓 영화 ",
  'link': 'http://v.daum.net/v/20260813091134224'},
 {'no': 14,
  'title': ' “우리 증조부도 매국노, 조상 업보 청산중”…영화 ‘암살’ 이경영 후손의 고백 ',
  'link': 'http://v.daum.net/v/20260813094909942'},
 {'no': 15,
  'title': ' 9월 준공 앞둔 부산기장촬영소…영화·관광 연계 시동 ',
  'link': 'http://v.daum.net/v/20260813084312164'},
 {'no': 16,
  'title': " '오디세이' 260만 돌파 1위…韓영화 1위 '오케이 마담2' ",
  'link': 'http://v.daum.net/v/20260813104857166'},
 {'no': 17,
  'title': ' 여름엔 할리우드 대작…韓 영화는 9월 승부수 [SS무비] ',
  'link': 'http://v.daum.net/v/20260813063159578'},
 {'no': 18,
  'title': ' 노들섬 ‘모기장 극장’서 강바람 쐬며 영화보세요 ',
  'link': 'http://v.

In [12]:
r = []
r.extend([1, 2, 3])
r.extend([4, 5, 6])
r

[1, 2, 3, 4, 5, 6]

In [13]:
result = [] # 해당 키워드 원하는 페이지 수만큼 검색한 결과를 담을 변수 dict 리스트
pages = 2
for page in range(1, pages+1):
    print(f'== {page} 페이지 수집 중 ==')
    item_result = collect_list('오디세이', page)
    result.extend(item_result)
    time.sleep(3)
pd.DataFrame(result)

== 1 페이지 수집 중 ==
== 2 페이지 수집 중 ==


,no,title,link
0,0,'오디세이' 무서운 예매율 역주행…90만장 돌파,http://v.daum.net/v/20260813082602648
1,1,"'오디세이' 놀란 감독, IMAX 매진 사태에 ""일반관서도 충분, 최상의 경험 가...",http://v.daum.net/v/20260812220130163
2,2,"오디세이 '용아맥' 암표가 12만원...놀란 감독 ""일반관도 충분""",http://v.daum.net/v/20260813104936197
3,3,'오디세이' 3시간인데...관객 몰리는 이유는?,http://v.daum.net/v/20260810130410816
4,4,"""웃돈 주고 새벽에 본다""...'오디세이' 예매 전쟁",http://v.daum.net/v/20260813004612990
5,5,"‘오디세이’ 암표 30만원까지…아이맥스관, 전쟁 같은 ‘피케팅’",http://v.daum.net/v/20260813144130103
6,6,'오디세이' 예매량 80만 넘었다…개봉 전보다 더 뜨거운 흥행세,http://v.daum.net/v/20260812141609437
7,7,'오디세이' 예매량 90만 돌파…극장가 사로잡은 제작 비하인드,http://v.daum.net/v/20260813102209722
8,8,"260만 ‘오디세이’, 예매만 90만 독주…‘오케이 마담2’는 4위[MK박스오피스]",http://v.daum.net/v/20260813091505365
9,9,"'예매 전쟁' 오디세이 앞줄서 봤다가...""3시간 내내 턱만 보였다""? [앵커리포트]",http://v.daum.net/v/20260813151033908


In [14]:
keywords = ['톰홀랜드', '영화']
pages = 3
result0 = [] # keyword[0] 1~pages페이지까지 검색한 결과 dict list
result1 = [] # keyword[1] 1~pages페이지까지 검색한 결과 dict list
for i, keyword in enumerate(keywords):
    print(f'= = {i+1}번째 검색어 {keyword} 검색 결과 수집({pages}페이지) 중입니다 = =')
    for page in range(1, pages+1):
        if i==0:
            result0.extend(collect_list(keyword, page))
        else:
            result1.extend(collect_list(keyword, page))
        time.sleep(3)

= = 1번째 검색어 톰홀랜드 검색 결과 수집(3페이지) 중입니다 = =
= = 2번째 검색어 영화 검색 결과 수집(3페이지) 중입니다 = =


In [15]:
result0_df = pd.DataFrame(result0)
result1_df = pd.DataFrame(result1)
result0_df.sample()

,no,title,link
7,7,톰 홀랜드♥젠데이아..부부 동반 극장가 쌍끌이 중 [스타이슈],http://v.daum.net/v/20260809080137469


In [16]:
result1_df.head()

,no,title,link
0,0,여름 대작 피했더니 9월에 6편 몰린 한국영화… ‘공멸’ 잔혹사 끊을까 [영화 뷰],http://v.daum.net/v/20260812082840601
1,1,“영화 찍고 기장에 머문다”… 기장군 ‘영화도시’ 시동,http://v.daum.net/v/20260811145750613
2,2,[썰물밀물] 영화와 도시개발,http://v.daum.net/v/20260813153006920
3,3,"김도훈→전종서, 넷플릭스 영화 '도차비' 출연 확정",http://v.daum.net/v/20260813110355056
4,4,엄정화 주연 '오케이 마담' 속편 개봉…이번 주 영화 소식,http://v.daum.net/v/20260813124510611


In [46]:
result0_df.to_csv(f'data/ch14_{keyword[0]}.csv', index=False, encoding='cp949')
result1_df.to_csv(f'data/ch14_{keyword[1]}.csv', index=False, encoding='cp949')

### 4) User-Agent를 추가하여 크롤링
- request.get(url), urlopen(url)함수를 사용하면 크롤링이 막혀있는 사이트
- 방법2에서 User-Agent를 추가하여 크롤링

- https://www.melon.com/robots.txt에서 일부 경로는 User-Agent에 봇이 지정

In [51]:
# 방법 1 
import requests
from bs4 import BeautifulSoup
url = 'https://www.melon.com/chart/'
melonResponse = requests.get(url)
print(melonResponse.status_code)
soup = BeautifulSoup(melonResponse.text, 'html.parser')
soup

406


In [52]:
# 방법2
from urllib.request import urlopen
from bs4 import BeautifulSoup
url = 'https://www.melon.com/chart/'
# melonResponse = urlopen(url) # HTTPError: HTTP Error 406: Not Accepatable

In [58]:
# User-Agent를 추가하여 방법2
from urllib.request import urlopen
from bs4 import BeautifulSoup
url = 'https://www.melon.com/chart/'
headrs = {'user-agent':
          'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36'}
# melonpage = Request(url, headers=headers)
melonpage = Request(url)
melonpage.add_header('user-agent',
                    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36')
melonResponse = urlopen(melonpage)
print(melonResponse.status)
soup = BeautifulSoup(melonResponse, 'html.parser')
# soup

200


In [59]:
# User-Agent를 추가하여 방법1
import requests
from bs4 import BeautifulSoup
url = 'https://www.melon.com/chart/'
headers = {'user-agent':
          'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36'}
melonResponse = requests.get(url, headers=headers)
print(melonResponse.status_code)
soup = BeautifulSoup(melonResponse.text, 'html.parser')
# soup

200


In [71]:
# 1위 : LOVE ATTACK | RESCENE (리센느)의 url {}
# 순위, 곡명, 가수, 가수페이지
rank_els = soup.select('div.wrap.t_center > span.rank')[1:] # 맨앞 '순위' 생략
ranks = [rank.text for rank in rank_els]

title_els = soup.select('div.ellipsis.rank01 > span > a')
titles = [title.text for title in title_els]

singer_els = soup.select('span.checkEllipsis') # 40위, 90위 가수가 복수명
singers = [singer.text.replace('/xa0','') for singer in singer_els]

links = []
for singer_el in singer_els:
    # print(singer_el)
    singer_link = 'https://www.melon.com' + singer_el.find('a').attrs.get('href')
    links.append(singer_link)
len(ranks), len(titles), len(singers), len(links)

# for idx, (title, singer, link) in enumerate(zip(titles, singers, links)):
#   print("{}위. {} | {}".format(idx+1, title, singer)) 
melon_chat_list = []
for rank, title, singer, link in zip(ranks, titles, singers, links):
    # print(f'{rank}위 {title} | {singer}')
    melon_chat_list.append({
        '순위':rank,
        '곡명':title,
        '가수':singer,
        '가수페이지':link
    })
pd.DataFrame(melon_chat_list) # pd.options.display.max_rows(60)행이상은 중간이 생략

,순위,곡명,가수,가수페이지
0,1,LOVE ATTACK,RESCENE (리센느),https://www.melon.com/artist/detail.htm?artist...
1,2,갑자기,아이오아이 (I.O.I),https://www.melon.com/artist/detail.htm?artist...
2,3,REDRED,CORTIS (코르티스),https://www.melon.com/artist/detail.htm?artist...
3,4,LEMONADE,aespa,https://www.melon.com/artist/detail.htm?artist...
4,5,Pretty Girl,RESCENE (리센느),https://www.melon.com/artist/detail.htm?artist...
...,...,...,...,...
95,96,BLACKHOLE,IVE (아이브),https://www.melon.com/artist/detail.htm?artist...
96,97,FOCUS,Hearts2Hearts (하츠투하츠),https://www.melon.com/artist/detail.htm?artist...
97,98,Soda Pop,"KPop Demon Hunters Cast, Danny Chung, Saja Boy...",https://www.melon.com/artist/detail.htm?artist...
98,99,OVERDRIVE,TWS (투어스),https://www.melon.com/artist/detail.htm?artist...


### 5) 네이버 지식인 검색(openAPI 사용X)
- 특정 keyword를 특정 페이지 수만큼

In [78]:
# 방법1 
from requests import get
from bs4 import BeautifulStoneSoup
keyword = '오디세이'
url = f'https://kin.naver.com/search/list.naver?query={keyword}'
print(url)
response = get(url)
print(response.status_code)
soup = BeautifulSoup(response.text, 'html.parser')

https://kin.naver.com/search/list.naver?query=오디세이
200


In [79]:
# 방법 2 
from urllib.request import urlopen
from bs4 import BeautifulSoup
from urllib.parse import quote
keyword = quote('오디세이')
url = f'https://kin.naver.com/search/list.naver?query={keyword}'
print(url)
response = urlopen(url)
print(response.status)
soup = BeautifulSoup(response, 'html.parser')

https://kin.naver.com/search/list.naver?query=%EC%98%A4%EB%94%94%EC%84%B8%EC%9D%B4
200


In [87]:
# keyword를 원하는 페이지 수만큼 
# 방법1 
from requests import get
from bs4 import BeautifulSoup
keyword = '오디세이'
pages = 2
items_list = [] # 크롤링한 데이터를 담을 list
for page in range(1, pages+1):
    url = f'https://kin.naver.com/search/list.naver?query={keyword}&page={page}'
    # print(url)
    # url = 'http://kin.naver.com/search/list.naver'
    # params = {'query':keyword, 'page':page}
    # response = get(url, params=params)
    response = get(url)
    # print(response.status_code)
    soup = BeautifulSoup(response.text, 'html.parser')
    # 글제목, link
    dt_els = soup.find_all('dt')
    for dt_el in dt_els:
        item = dt_el.find('a')
        items_list.append({
            'title':item.text,
            'link':item.attrs.get('href')
        })
df = pd.DataFrame(items_list)

In [108]:
df

,순위,책이름,저자,출판사,가격 정보
0,1,"세네카, 오늘을 빼앗기고 있는 당신에게",루키우스 안나이우스 세네카,논픽션,"16,200"
1,2,오디세이아,하와이 대저택,현대지성,"24,300"
2,3,원소 원정대: 118개 캐릭터로 마스터하는 주기율표 공략집,호메로스,윌북주니어,"17,820"
3,4,오뒷세이아,페테르 파울 루벤스,을유문화사,"22,500"
4,5,싯다르타,박문재,민음사,"7,200"
5,6,투명한 나선,아게도리도리,북다,"19,800"
6,7,니체의 초월자,박재현,히읏,"15,210"
7,8,오디세이아,장홍제,마음이음,"15,120"
8,9,모순,호메로스,쓰다,"11,700"
9,10,최상위권의 비밀,김헌,동양북스(동양books),"17,820"


## 2.2 open API 사용 : json 웹데이터 수집
### 1) 네이버 지식인으로 검색 (open API 사용 O)

- 네이버개발자센터에서 어플리케이션 등록(id, pw) => NAVER API HUB

In [89]:
%pip install dotenv

  Using cached dotenv-0.9.9-py2.py3-none-any.whl (1.9 kB)
Note: you may need to restart the kernel to use updated packages.


In [1]:
# 환경변수를 쓰기 위한 패키지 : dotenv
from dotenv import load_dotenv
import os
load_dotenv(
    #dotenv_path='.env'
)
print(os.getenv('CLIENT_ID')[:3])
print(os.getenv('CLIENT_SECRET')[:3])

wCt
Zry


In [6]:
# 방법 2
import os
import sys
import urllib.request
import json
client_id = os.getenv('CLIENT_ID')
client_secret = os.getenv('CLIENT_SECRET')
encText = urllib.parse.quote("오디세이")
url = f"https://openapi.naver.com/v1/search/kin.json?query={encText}" # JSON 결과
# request = urllib.request.Request(url)
# request.add_header("X-Naver-Client-Id",client_id)
# request.add_header("X-Naver-Client-Secret",client_secret)
headers = {
    'X-Naver-Client-Id': client_id,
    'X-Naver-Client-Secret':client_secret
}
request = urllib.request.Request(url, headers=headers)
response = urllib.request.urlopen(request)
rescode = response.getcode()
if(rescode==200):
    response_body = response.read()
    # print(response_body.decode('utf-8')[:30])
else:
    print("Error Code:" + rescode)

data = json.loads(response_body) # 문자를 json형태로 변환
print('data의 응답들 :', data.keys())
print('검색결과 갯수 :', len(data['items']))

items = data['items']
items_list = []
for item in items:
    # print(item)
    title = item['title'].replace('<b>', '').replace('</b>','')
    link = item['link']
    description = item['description'].replace('<b>','').replace('</b>','')
    items_list.append([title, link, description])
pd.DataFrame(items_list, columns=['title', 'link', 'description']).sample()

data의 응답들 : dict_keys(['lastBuildDate', 'total', 'start', 'display', 'items'])
검색결과 갯수 : 10


,title,link,description
5,"'오디세이' 영화 속 캐릭터, 실제 배우들과 닮았을까요?",https://kin.naver.com/qna/detail.naver?dirId=3...,"'오디세이'라는 영화를 알게 되었는데, 주연 배우들이 맡은 캐릭터가 실제 배우들의 ..."


In [7]:
# 방법 1 
import os
import requests
import json
client_id = os.getenv('CLIENT_ID')
client_secret = os.getenv('CLIENT_SECRET')
encText = "오디세이"
url = f"https://openapi.naver.com/v1/search/kin.json" # JSON 결과
params = {'query':encText, 'display':20 }
headers = {
    'X-Naver-Client-Id': client_id,
    'X-Naver-Client-Secret':client_secret
}
response = requests.get(url, params=params, headers=headers)
# print('response 상태 :', response.status_code)
# print(response.text[:30])
# data = json.loads(response.text) # 문자를 json형태로 변환
data = response.json()['items'] # response를 json형태로 변환한 것중 'items'
items_list = []
for item in items:
    # print(item)
    title = item['title'].replace('<b>', '').replace('</b>','')
    link = item['link']
    description = item['description'].replace('<b>','').replace('</b>','')
    items_list.append([title, link, description])
pd.DataFrame(items_list, columns=['title', 'link', 'description']).sample()

,title,link,description
6,오디세이 용아맥 자리,https://kin.naver.com/qna/detail.naver?dirId=8...,오디세이 첫주차 예매 하긴 했는데 m40이라 취소하고 다른 표를 노릴까 생각중이에요...


### NAVER API HUB 방식으로 지식IN 검색

In [21]:
# 방법 1 
import os
import requests
import json
import pandas as pd
from dotenv import load_dotenv
load_dotenv() # 환경 변수 load
client_id = os.getenv('NEW_CLIENT_ID')
client_secret = os.getenv('NEW_CLIENT_SECRET')
encText = "광주"
url = f"https://naverapihub.apigw.ntruss.com/search/v1/kin" # JSON 결과
params = {'query':encText, 'display':20 }
headers = {
    'X-NCP-APIGW-API-KEY-ID': client_id,
    'X-NCP-APIGW-API-KEY':client_secret
}
response = requests.get(url, params=params, headers=headers)

items = response.json()['items'] # response를 json형태로 변환한 것중 'items'
items_list = []
for item in items:
    # print(item)
    title = item['title'].replace('<b>', '').replace('</b>','')
    link = item['link']
    description = item['description'].replace('<b>','').replace('</b>','')
    items_list.append([title, link, description])
pd.DataFrame(items_list, columns=['title', 'link', 'description']).sample()

,title,link,description
19,광주 피부과,https://kin.naver.com/qna/detail.naver?dirId=7...,많은데 광주에 공장형 피부과 말고 피부 상담 후에 압출이랑 시술 추천 받고 시술하고...


### quiz) 네이버 open API를 이용하여 원하는 query 이미지 100건의 데이터를 'data/img_list.csv'파일로
- title(제목), link(링크), thumbnail(썸네일), sizeheight, sizewidth

In [20]:
def get_image_list(query):
    'query로 검색한 이미지 정보(제목, 링크,썸네일, size) 100건 데이터 프레임을 return(방법1)'
    from dotenv import load_dotenv
    import os
    import requests
    import pandas as pd
    load_dotenv()
    client_id = os.getenv('CLIENT_ID')
    client_secret = os.getenv('CLIENT_SECRET')
    headers = {
        'X-Naver-Client-Id': client_id,
        'X-Naver-Client-Secret':client_secret
    }
    url = 'https://openapi.naver.com/v1/search/image'
    params = {'query':query, 'display':100 }
    response = requests.get(url, params=params, headers=headers)
    
    items = response.json()['items']
    # print(items[:2])
    items_list = []
    for item in items:
        items_list.append({
            '제목':item.get('title'),
            '링크':item.get('link'),
            '썸네일':item.get('thumbnail'),
            'sizeheight':item.get('sizeheight'),
            'sizewidth':item.get('sizewidth')
        })
    return pd.DataFrame(items_list)
get_image_list("청바지")    

,제목,링크,썸네일,sizeheight,sizewidth
0,[홍은]통바지 데님 연청 여자 하이웨이스트 와이드 청바지 | 텐바이텐,https://thumbnail.10x10.co.kr/webimage/image/b...,https://search.pstatic.net/sunny/?type=b150&sr...,500,500
1,여성용 와이드 하이웨스트 데님팬츠 데일리 청바지 5516,http://shopping.phinf.naver.net/main_5712970/5...,https://search.pstatic.net/common/?type=b150&s...,800,800
2,가을 겨울 일자핏 캐주얼 와이드 청바지 DNZK03 : 스마트 공장,http://shop1.phinf.naver.net/20241125_266/1732...,https://search.pstatic.net/common/?type=b150&s...,916,750
3,여자 스키니진 스판 타이트 청바지 | 텐바이텐,https://thumbnail.10x10.co.kr/webimage/image/b...,https://search.pstatic.net/sunny/?type=b150&sr...,500,500
4,남자 와이드 스랙스 9부 바지 가 남성 청바지 루즈한 스트레이트 리 광저우 : so...,http://shop1.phinf.naver.net/20260331_151/1774...,https://search.pstatic.net/common/?type=b150&s...,1000,750
...,...,...,...,...,...
95,MOSAIRATION 여성 청바지 하이웨스트 캐주얼 스트레이트 데님 팬츠 M0011214,http://shopping.phinf.naver.net/main_5905233/5...,https://search.pstatic.net/common/?type=b150&s...,1000,1000
96,여성 바지 빈티지 캐주얼 가을 청바지 데님바지 엘보 배기핏 와이드 워싱 데일룩 DA...,https://shop-phinf.pstatic.net/20260225_102/17...,https://search.pstatic.net/common/?type=b150&s...,500,500
97,여성 슬림핏 하이웨이스트 데님 청바지 봄 와이드 스트레이트 크롭 : 하유유통,https://shop-phinf.pstatic.net/20260605_15/178...,https://search.pstatic.net/common/?type=b150&s...,800,800
98,타미힐피거 기본 워싱 청바지 EMM3PD91A,http://shopping.phinf.naver.net/main_6623271/6...,https://search.pstatic.net/common/?type=b150&s...,500,500


In [22]:
df.to_csv('data/img_list.csv', encoding='cp949', index=False)

In [23]:
# df에 있는 이미지 로컬에 저장하기
print(df.loc[1, '링크']) # 메인_01_청바지.jpg
print(df.loc[1, '썸네일']) # 썸네일_01_청바지.jpg

http://shopping.phinf.naver.net/main_5712970/57129708867.20251008173842.jpg
https://search.pstatic.net/common/?type=b150&src=http%3A%2F%2Fshopping.phinf.naver.net%2Fmain_5712970%2F57129708867.20251008173842.jpg


In [26]:
import pandas as pd
df = pd.read_csv('data/img_list.csv', encoding='cp949')

In [27]:
df = df[df['링크'].str.find('?')==-1]
df

,제목,링크,썸네일,sizeheight,sizewidth
1,여성용 와이드 하이웨스트 데님팬츠 데일리 청바지 5516,http://shopping.phinf.naver.net/main_5712970/5...,https://search.pstatic.net/common/?type=b150&s...,800,800
2,가을 겨울 일자핏 캐주얼 와이드 청바지 DNZK03 : 스마트 공장,http://shop1.phinf.naver.net/20241125_266/1732...,https://search.pstatic.net/common/?type=b150&s...,916,750
4,남자 와이드 스랙스 9부 바지 가 남성 청바지 루즈한 스트레이트 리 광저우 : so...,http://shop1.phinf.naver.net/20260331_151/1774...,https://search.pstatic.net/common/?type=b150&s...,1000,750
5,남자밴딩청바지 세미와이드청바지 통청바지 : 윤팬츠,http://shop1.phinf.naver.net/20250902_117/1756...,https://search.pstatic.net/common/?type=b150&s...,1000,1026
6,지앤제이 라이크 슬림 일자진 여자청바지 청바지,https://cdn2.halfclub.com/cdn/product/A7548/P3...,https://search.pstatic.net/sunny/?type=b150&sr...,640,640
...,...,...,...,...,...
95,MOSAIRATION 여성 청바지 하이웨스트 캐주얼 스트레이트 데님 팬츠 M0011214,http://shopping.phinf.naver.net/main_5905233/5...,https://search.pstatic.net/common/?type=b150&s...,1000,1000
96,여성 바지 빈티지 캐주얼 가을 청바지 데님바지 엘보 배기핏 와이드 워싱 데일룩 DA...,https://shop-phinf.pstatic.net/20260225_102/17...,https://search.pstatic.net/common/?type=b150&s...,500,500
97,여성 슬림핏 하이웨이스트 데님 청바지 봄 와이드 스트레이트 크롭 : 하유유통,https://shop-phinf.pstatic.net/20260605_15/178...,https://search.pstatic.net/common/?type=b150&s...,800,800
98,타미힐피거 기본 워싱 청바지 EMM3PD91A,http://shopping.phinf.naver.net/main_6623271/6...,https://search.pstatic.net/common/?type=b150&s...,500,500


In [24]:
url = "http://a.net/a.jpg"
# '.'+url.split('.')[-1]
url[url.rfind('.'):]

'.jpg'

In [25]:
def save_image(attr, idx, link, query):
    'link의 이미지를 image/attr_idx_query.확장자로 local에 저장'
    import requests, os
    from urllib.parse import urlparse
    response = requests.get(link)
    # link에서 확장자(jpg) 추출
    file_extension = link.split('.')[-1]
    # 확장자 뒤에 ?가 있는 위치(?가 없으면 -1)
    index = file_extension.find('?') 
    if index != -1:
        file_extension = file_extension[:index]
    # 확장자 뒤에 %가 있는 위치
    index = file_extension.find('%')
    if index != -1:
        file_extension = file_extension[:index]
    # 허용할 이미지 확장자인지
    valid_extension = ['jpg', 'jpeg', 'png', 'gif', 'bmp', 'webp', 'svg']
    if file_extension.lower() not in valid_extension:
        # 허용할 이미지 확장자가 아닌 경우(ex: .net)
        print(f'{idx}번째 {attr}의 url에서 확장자를 추출 못하여 header에서 도전해봄')
        content_type = response.headers.get('Content-Type','') # image/jpeg
        file_extension = content_type.split('image/')[-1] # jpeg
        file_extension = file_extension.replace('jpeg', 'jpg')
        file_extension = file_extension if file_extension in valid_extension else 'jpg'
    # image 폴더가 없으면 image 폴더 생성
    save_dir = 'image'
    os.makedirs(save_dir, exist_ok=True)
    # 이미지 저장
    with open(f'{save_dir}/{attr}_{idx:02}_{query}.{file_extension}', 'wb') as f:
        f.write(response.content) # response의 바이너리를 저장
save_image('썸네일', 0, df.loc[0,'썸네일'], '청바지')

In [ ]:
# naver api 요청받아 이미지 제목, link, 썸네일 link 정보를 csv로 백업, 이미지 link와 썸네일 link 이미지를 다운

In [31]:
def get_image_list_save_file(query):
    '''
    naver api 요청받아 이미지 제목, link, 썸네일 link 정보를 데이터프레임으로 return
    데이터프레임을 csv로 백업
    이미지link와 썸네일link 이미지를 다운(방법2)
    '''
    from dotenv import load_dotenv
    import os
    from urllib.request import urlopen, Request
    from urllib.parse import quote
    import json
    import pandas as pd
    load_dotenv(dotenv_path='d:/.env')
    client_id = os.getenv('CLIENT_ID')
    client_secret = os.getenv('CLIENT_SECRET')
    headers = {
        'X-Naver-Client-Id': client_id,
        'X-Naver-Client-Secret':client_secret
    }
    query = quote(query)
    url = f'https://openapi.naver.com/v1/search/image?query={query}&display=100'
    request = Request(url, headers=headers)
    response = urlopen(request)
    
    items = json.loads(response.read())['items']
    items_list = []
    for idx, item in enumerate(items):
        link = item.get('link')
        thumbnail = item.get('thumbnail')
        items_list.append({
            '제목':item.get('title'),
            '링크':link,
            '썸네일':thumbnail,
            'sizeheight':item.get('sizeheight'),
            'sizewidth':item.get('sizewidth')
        })
        # 이미지 저장 save_image('메인', idx, link, query) save_image('썸네일', idx, thumbnail, query) 
        save_image('메인', idx, link, query)
        save_image('썸네일', idx, thumbnail, query)
        if idx%10==0:
            print(f'==={idx}% 진행완료===')
    result = pd.DataFrame(items_list)
    result.to_csv('image/img_list.csv', index=False)
    print('이미지 및 csv 저장완료')
    return result

In [33]:
df = get_image_list_save_file("청바지")

===0% 진행완료===
===10% 진행완료===
===20% 진행완료===
===30% 진행완료===
===40% 진행완료===
===50% 진행완료===
===60% 진행완료===
===70% 진행완료===
===80% 진행완료===
===90% 진행완료===
이미지 및 csv 저장완료


## 2.3 XML 웹 데이터 수집
- RSS서비스 , openAPI사용
### 1) 전국 날씨 RSS를 BeautifulSoup을 이용한 xml 크롤링
- 기상청 RSS에서 1개월 기상정보를 서비스

In [47]:
import requests 
# from urllib.request import urlopen
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
items_list = []
url = 'https://www.kma.go.kr/repositary/xml/fct/mon/img/fct_mon1rss_108_20260813.xml'
target = requests.get(url)
soup = BeautifulSoup(target.text, 'xml')
locals = soup.select('local_ta')
# print(locals[1])
for local in locals:
    local_name = local.select_one('local_ta_name').text.strip()
    week1_local_ta_normalYear = local.select_one('week1_local_ta_normalYear').text # 평년기온
    week1_local_ta_similarRange = local.select_one('week1_local_ta_similarRange').text # 예측범위
    week1_local_ta_minVal       = local.select_one('week1_local_ta_minVal').text # 평년보다 낮을 확률
    week1_local_ta_similarVal   = local.select_one('week1_local_ta_similarVal').text # 평년과 비슷할 확률
    week1_local_ta_maxVal       = local.select_one('week1_local_ta_maxVal').text # 평년보다 높을 확률
    items_list.append({
        '지역': local_name,
        '평년기온':week1_local_ta_normalYear,
        '예측범위':week1_local_ta_similarRange,
        '낮을확률':week1_local_ta_minVal,
        '같을확률':week1_local_ta_similarVal,
        '높을확률':week1_local_ta_maxVal,
    })
df = pd.DataFrame(items_list)
df.head()

,지역,평년기온,예측범위,낮을확률,같을확률,높을확률
0,"전국(제주도,북한제외)",23.7,23.1~24.3,10,30,60
1,서울ㆍ인천ㆍ경기도,23.9,23.3~24.5,10,30,60
2,강원도 영서,22.1,21.4~22.8,10,30,60
3,강원도 영동,22.0,21.3~22.7,10,30,60
4,대전ㆍ세종ㆍ충청남도,24.0,23.4~24.6,10,30,60


In [48]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13 entries, 0 to 12
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   지역      13 non-null     object
 1   평년기온    13 non-null     object
 2   예측범위    13 non-null     object
 3   낮을확률    13 non-null     object
 4   같을확률    13 non-null     object
 5   높을확률    13 non-null     object
dtypes: object(6)
memory usage: 752.0+ bytes


In [51]:
df['평년기온'] = df['평년기온'].astype(np.float64)
df['낮을확률'] = df['낮을확률'].astype(np.int16)
df['같을확률'] = df['같을확률'].astype(np.int16)
df['높을확률'] = df['높을확률'].astype('int')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13 entries, 0 to 12
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   지역      13 non-null     object 
 1   평년기온    13 non-null     float64
 2   예측범위    13 non-null     object 
 3   낮을확률    13 non-null     int16  
 4   같을확률    13 non-null     int16  
 5   높을확률    13 non-null     int32  
dtypes: float64(1), int16(2), int32(1), object(2)
memory usage: 544.0+ bytes


### 2) xml로 응답하는 open API 활용
- data.go.kr에서
    * 서울특별시_노선정보조회 서비스(버스id, 버스 정류장 목록-정류장 id와 정류장이름)
    * 서울특별시_버스위치정보조회 서비스(실시간 버스 위치 목록)

In [1]:
# step 1 : 버스번호의 busRouteId 받아오기
# 서울특별시_노선정보조회 서비스 : 노선번호 목록조회(3번 . getBusRouteList)
# https://api.data.go.kr/contents/sub02/getBusPosByRtidList.html

In [1]:
# 인증키
from dotenv import load_dotenv
import os
load_dotenv()
serviceKey = os.getenv('key')
serviceKey

'3c93731ebcca9160b8239fdac02011dd266dc43ee3cbe34be807c31a6bc6c2a9'

In [19]:
import requests
from urllib.request import quote
from bs4 import BeautifulSoup
from urllib.request import urlretrieve # xml을 로컬에 저장
busNum = quote('마포01')
busNum = '162' 
key = os.getenv('key')
url1 = f'http://ws.bus.go.kr/api/rest/busRouteInfo/getBusRouteList?strSrch={busNum}&ServiceKey={key}'
print(url1)
# savefilename1='data/ch14_1.busInfo.xml'
# urlretrieve(url1, savefilename1)
# with open(savefilename1, encoding='utf-8') as f:
#     xml = f.read();
# soup = BeautifulSoup(xml, 'xml')
xml = requests.get(url1)
soup = BeautifulSoup(xml.text, 'xml')
# soup

http://ws.bus.go.kr/api/rest/busRouteInfo/getBusRouteList?ServiceKey=3c93731ebcca9160b8239fdac02011dd266dc43ee3cbe34be807c31a6bc6c2a9&strSrch=162&ServiceKey=3c93731ebcca9160b8239fdac02011dd266dc43ee3cbe34be807c31a6bc6c2a9


In [20]:
for item in soup.select('itemList'):
    busRouteNm = item.select_one('busRouteNm').text
    if busNum == busRouteNm:
        busRouteId = item.select_one('busRouteId').text
        break
print('busRouteId =', busRouteId)

busRouteId = 100100034


In [21]:
# step 2 : 해당 버스 busRouteId의 정류장 목록 받아오기(정류장id, 정류장이름)
# 서울특별시_노선정보조회 서비스 : 4번 기능(getStationByRoute) 이용
# https://api.bus.go.kr/contents/sub02/getStationByRoute.html

In [27]:
import pandas as pd
url2 = f'http://ws.bus.go.kr/api/rest/busRouteInfo/getStaionByRoute?ServiceKey={key}&busRouteId={busRouteId}'
print(url2)
response = requests.get(url2)
soup = BeautifulSoup(response.text, 'xml')
itemLists = soup.select('itemList')
print(f'{busNum}번 정류장 갯수 :', len(itemLists))
bus_station = []
for itemList in itemLists:
    stationNm = itemList.select_one('stationNm').text # 정류장명
    station   = itemList.select_one('station').text   # 정류장 ID
    gpsX      = itemList.select_one('gpsX').text # 경도
    gpsY      = itemList.select_one('gpsY').text # 위도
    bus_station.append([stationNm, station, gpsX, gpsY])
df_station = pd.DataFrame(bus_station, columns=['정류소명', 'id', '경도', '위도'])
df_station

http://ws.bus.go.kr/api/rest/busRouteInfo/getStaionByRoute?ServiceKey=3c93731ebcca9160b8239fdac02011dd266dc43ee3cbe34be807c31a6bc6c2a9&busRouteId=100100034
162번 정류장 갯수 : 77


,정류소명,id,경도,위도
0,정릉산장아파트,107000071,127.003343,37.616712
1,정릉4동주민센터.경국사,107000073,127.006345,37.613529
2,북한산보국문역2번출구,107000518,127.0079858233,37.612293899
3,성북청수도서관.정릉4동성당,107000075,127.0084193769,37.6115696748
4,정릉시장입구,107000077,127.0098212542,37.6084653256
...,...,...,...,...
72,성북청수도서관.정릉4동성당,107000076,127.009045,37.610876
73,북한산보국문역1번출구,107000519,127.008329146,37.6120835499
74,정릉4동주민센터.경국사,107000074,127.006681,37.613335
75,정릉대우아파트,107000072,127.00386,37.616708


In [ ]:
# step 3 : 해당 버스 busRouteId의 차량들의 위치정보 (차량번호, 혼잡도, 경도, 위도, 최종정류장id, 다음정류장id, 도착소요시간)
# 서울특별시_버스위치정보조회 서비스 - 2번 기능(getBusPosByRtidList) 이용
# http://ws.bus.go.kr/api/rest/buspos/getBusPosByRtid

In [41]:
url3 = f'http://ws.bus.go.kr/api/rest/buspos/getBusPosByRtid?serviceKey={key}&busRouteId={busRouteId}'
print(url3)
response = requests.get(url3)
soup = BeautifulSoup(response.text, 'xml')
itemLists = soup.select('itemList')
print(f'{busNum}번 운행중인 버스는 {len(itemLists)}대입니다')
bus_position = [] # 버스 위치정보를 담을 list
for itemList in itemLists:
    plainNo = itemList.select_one('plainNo').text # 차량번호
    congetion = itemList.select_one('congetion').text
    # 0:없음, 3:여유, 4:보통, 5:혼잡
    congetion = '없음' if congetion=='0' \
            else '여유' if congetion=='3' \
            else '보통' if congetion=='4' \
            else '혼잡' 
    gpsX = itemList.select_one('gpsX').text # 경도
    gpsY = itemList.select_one('gpsY').text # 위도
    lastStnId = itemList.select_one('lastStnId').text # 최종정류소id
    nextStId = itemList.select_one('nextStId').text # 다음정류소id
    nextStTm = itemList.select_one('nextStTm').text # 다음종류소도착소요시간
    bus_position.append({
        '차량번호':plainNo,
        '혼잡도':congetion,
        '경도':gpsX,
        '위도':gpsY,
        '최종정류소id':lastStnId,
        '다음정류소id':nextStId,
        '도착소요시간':nextStTm
    })
df_position = pd.DataFrame(bus_position)
df_position.head(2)

http://ws.bus.go.kr/api/rest/buspos/getBusPosByRtid?serviceKey=3c93731ebcca9160b8239fdac02011dd266dc43ee3cbe34be807c31a6bc6c2a9&busRouteId=100100034
162번 운행중인 버스는 24대입니다


,차량번호,혼잡도,경도,위도,최종정류소id,다음정류소id,도착소요시간
0,서울74사1638,없음,127.000678,37.617564,107000071,107000169,229
1,서울74사3360,여유,127.008419,37.61157,107000075,107000079,191


In [49]:
df_station.loc[df_station['id']=='107000169', '정류소명'].iloc[0]

'정릉입구.정릉역'

In [56]:
def get_station_name(row):
    row['최종정류소명'] = df_station.loc[df_station['id']==row['최종정류소id'], '정류소명'].iloc[0]
    row['다음정류소명'] = df_station.loc[df_station['id']==row['다음정류소id'], '정류소명'].iloc[0]
    return row

In [63]:
# get_station_name(df_position.iloc[0])
df_position = df_position.apply(get_station_name, axis=1)
df_position.head(1)

,차량번호,혼잡도,경도,위도,최종정류소id,다음정류소id,도착소요시간,최종정류소명,다음정류소명
0,서울74사1638,없음,127.000678,37.617564,107000071,107000169,229,정릉산장아파트,정릉입구.정릉역


In [65]:
drop_col = df_position.columns.str.contains('id')
drop_column_names = df_position.columns[drop_col]
df_position.drop(drop_column_names, axis=1, inplace=True)

In [70]:
df_position['도착소요시간'] = round( df_position['도착소요시간'].astype('int')/60, 2 )
df_position.head()

,차량번호,혼잡도,경도,위도,도착소요시간,최종정류소명,다음정류소명
0,서울74사1638,없음,127.000678,37.617564,3.82,정릉산장아파트,정릉입구.정릉역
1,서울74사3360,여유,127.008419,37.61157,3.18,성북청수도서관.정릉4동성당,정릉우체국앞
2,서울74사1625,여유,127.013579,37.60223,36.13,정릉입구.정릉역,소공동.롯데영플라자
3,서울74사2216,여유,127.01419,37.60113,36.12,정릉입구.정릉역,소공동.롯데영플라자
4,서울74사2218,여유,127.016173,37.593891,30.82,성신여대입구역5번출구,소공동.롯데영플라자


# 3절. 연습문제
    - YES24 베스트셀러 정보에서
     순위1~48위까지의 "순위, 책이름, 저자, 출판사, 가격 정보를 출력하고,
     ch14_yes24_bestseller.csv로 백업

In [72]:
import requests 
from urllib.request import urlopen
from bs4 import BeautifulSoup
import pandas as pd

In [124]:
pages = 2
yes24_bestseller_list = []
for page in range(1, pages+1):
    url = f'https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageNumber={page}'
 
    Response = requests.get(url)    
    soup = BeautifulSoup(Response.text, 'html.parser')

    rank_els = soup.select('div.img_upper > em.ico.rank')
    ranks = [rank.text.strip() for rank in rank_els]

    title_els = soup.select('a.gd_name')
    titles = [title.text.strip() for title in title_els]

    author_els = soup.select('span.authPub.info_auth > a') 
    authors = [author.text for author in author_els]

    publisher_els = soup.select('span.authPub.info_pub > a') 
    publishers = [publisher.text for publisher in publisher_els]

    price_els = soup.select('strong.txt_num > em.yes_b') 
    prices = [price.text for price in price_els]

    
    for rank, title, author, publisher, price in zip(ranks, titles, authors, publishers, prices):
            yes24_bestseller_list.append({
                '순위':rank,
                '책이름':title,
                '저자':author,
                '출판사':publisher,
                '가격 정보':price
            })
df = pd.DataFrame(yes24_bestseller_list)
df

,순위,책이름,저자,출판사,가격 정보
0,1,"세네카, 오늘을 빼앗기고 있는 당신에게",루키우스 안나이우스 세네카,논픽션,"16,200"
1,2,오디세이아,하와이 대저택,현대지성,"24,300"
2,3,원소 원정대: 118개 캐릭터로 마스터하는 주기율표 공략집,호메로스,윌북주니어,"17,820"
3,4,오뒷세이아,페테르 파울 루벤스,을유문화사,"22,500"
4,5,싯다르타,박문재,민음사,"7,200"
5,6,투명한 나선,아게도리도리,북다,"19,800"
6,7,니체의 초월자,박재현,히읏,"15,210"
7,8,오디세이아,장홍제,마음이음,"15,120"
8,9,모순,호메로스,쓰다,"11,700"
9,10,최상위권의 비밀,김헌,동양북스(동양books),"17,820"


In [125]:
df.to_csv('ch14_yes24_bestseller.csv', index=False, encoding='utf-8')